In [1]:
import json
import os
import requests

from tqdm.auto import tqdm

In [2]:
docs_url = 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/documents.json?raw=1'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [24]:
documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [13]:
from elasticsearch import Elasticsearch
from elasticsearch import NotFoundError

es = Elasticsearch('http://localhost:9200',   request_timeout=60)

INDEX_NAME =  "course-questions"

index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"}
        }
    }
}

In [18]:
try:
    es.indices.get(index=INDEX_NAME)
    print(f"{INDEX_NAME} already exists")
    # Використання options() для передачі транспортних параметрів
    es.options(ignore_status=[400, 404]).indices.delete(index=INDEX_NAME)
except NotFoundError:
    response = es.indices.create(index=INDEX_NAME,
                                           settings=index_settings['settings'],
                                           mappings=index_settings['mappings'])
    print(response)

{'acknowledged': True, 'shards_acknowledged': False, 'index': 'course-questions'}


In [19]:
# Отримати список індексів
!curl -X GET "localhost:9200/_cat/indices?v"

health status index            uuid                   pri rep docs.count docs.deleted store.size pri.store.size
red    open   course-questions AivVzt6sSbamhj27V2ZZjA   1   0                                                  


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100   224  100   224    0     0  10476      0 --:--:-- --:--:-- --:--:-- 10666


In [20]:
# переглянути інформацію про вузли
!curl -X GET "localhost:9200/_cat/nodes?v"

ip         heap.percent ram.percent cpu load_1m load_5m load_15m node.role   master name
172.24.0.2            5          57   0    0.08    0.08     0.07 cdfhilmrstw *      52928730c32b


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100   186  100   186    0     0   5851      0 --:--:-- --:--:-- --:--:--  6000


In [21]:
# Проиндексируйте документы
for doc in tqdm(documents):
    es.index(index=INDEX_NAME, document=doc)

print("Indexing complete.")

  0%|          | 0/948 [00:00<?, ?it/s]

ConnectionTimeout: Connection timed out

In [23]:
from pprint import pprint

# Определите поисковый запрос
search_query = {
    "query": {
        "bool": {
            "should": [
                {"match": {"question": {"query": "How do I execute a command in a running docker container?", "boost": 4}}},
                {"match": {"text": "How do I execute a command in a running docker container?"}}
            ]
        }
    }
}

# Выполните поиск
response = es.search(index=INDEX_NAME, query=search_query['query'])

# Выведите топовый результат
top_result = response['hits']['hits'][0]
pprint(top_result)
print(f"Top result score: {top_result['_score']}")

ApiError: ApiError(503, 'search_phase_execution_exception', None)

In [5]:
import pandas as pd

df = pd.DataFrame(documents, columns=['course', 'section', 'question', 'text'])
df.head()

,course,section,question,text
0,data-engineering-zoomcamp,General course-related questions,Course - When will the course start?,The purpose of this document is to capture fre...
1,data-engineering-zoomcamp,General course-related questions,Course - What are the prerequisites for this c...,GitHub - DataTalksClub data-engineering-zoomca...
2,data-engineering-zoomcamp,General course-related questions,Course - Can I still join the course after the...,"Yes, even if you don't register, you're still ..."
3,data-engineering-zoomcamp,General course-related questions,Course - I have registered for the Data Engine...,You don't need it. You're accepted. You can al...
4,data-engineering-zoomcamp,General course-related questions,Course - What can I do before the course starts?,You can start by installing and setting up all...


In [3]:
context_template = """
Q: {question}
A: {text}
""".strip()

prompt_template = """

You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

In [4]:
len(prompt_template)

206

In [9]:
import tiktoken
encoding = tiktoken.encoding_for_model("gpt-4o")

ModuleNotFoundError: No module named 'tiktoken'